# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
from dotenv import load_dotenv
import os
os.environ["data_prices_path"] = r"C:\Users\werne\OneDrive\Desktop\School Apps\Production\production-Assignment-1\05_src\data\prices"
# Load environment variables  
load_dotenv()


True

In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.
#Get PRICE_DATA_PATH from .env file
PRICE_DATA_PATH = r"C:\Users\werne\OneDrive\Desktop\School Apps\Production\production-Assignment-1\05_src\data\prices"

#Find path of all parquet files in PRICE_DATA_PATH
parquet_files = glob(os.path.join(PRICE_DATA_PATH, "**", "*.parquet"), recursive=True)  

#Print the path of all parquet files
print(parquet_files)


['C:\\Users\\werne\\OneDrive\\Desktop\\School Apps\\Production\\production-Assignment-1\\05_src\\data\\prices\\ACN\\ACN_2001\\part.0.parquet', 'C:\\Users\\werne\\OneDrive\\Desktop\\School Apps\\Production\\production-Assignment-1\\05_src\\data\\prices\\ACN\\ACN_2001\\part.1.parquet', 'C:\\Users\\werne\\OneDrive\\Desktop\\School Apps\\Production\\production-Assignment-1\\05_src\\data\\prices\\ACN\\ACN_2002\\part.0.parquet', 'C:\\Users\\werne\\OneDrive\\Desktop\\School Apps\\Production\\production-Assignment-1\\05_src\\data\\prices\\ACN\\ACN_2002\\part.1.parquet', 'C:\\Users\\werne\\OneDrive\\Desktop\\School Apps\\Production\\production-Assignment-1\\05_src\\data\\prices\\ACN\\ACN_2003\\part.0.parquet', 'C:\\Users\\werne\\OneDrive\\Desktop\\School Apps\\Production\\production-Assignment-1\\05_src\\data\\prices\\ACN\\ACN_2003\\part.1.parquet', 'C:\\Users\\werne\\OneDrive\\Desktop\\School Apps\\Production\\production-Assignment-1\\05_src\\data\\prices\\ACN\\ACN_2004\\part.0.parquet', 'C:\\

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [4]:
# Write your code below.
import dask.dataframe as dd

#Add lags for variables close and Adj_Close
dd_raw = dd.read_parquet(parquet_files).set_index("ticker")
#Examine the columns of the dataframe - Was receiving Syntax error on spelling of names of columns
print(dd_raw.columns)


Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'Year'],
      dtype='object')


In [5]:
def add_features_partition(df):
    df = df.sort_values(["ticker", "Date"])
    df["close_lag1"] = df.groupby("ticker")["Close"].shift(1)
    df["close_lag2"] = df.groupby("ticker")["Close"].shift(2)
    df["close_lag3"] = df.groupby("ticker")["Close"].shift(3)

    df["adj_close_lag1"] = df.groupby("ticker")["Adj Close"].shift(1)
    df["adj_close_lag2"] = df.groupby("ticker")["Adj Close"].shift(2)
    df["adj_close_lag3"] = df.groupby("ticker")["Adj Close"].shift(3)

    df["returns"] = (df["Close"] / df["close_lag1"]) - 1

    df["hi_lo_range"] = df["High"] - df["Low"]
    return df

#Assign the result to `dd_feat`.
dd_feat = dd_raw.map_partitions(add_features_partition)

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [6]:
# Write your code below.
#Convert the Dask DataFrame to a Pandas DataFrame
pdf = dd_feat.compute() 
#Contain the moving average of returns using 10 day window
pdf["returns_ma10"] = pdf.groupby("ticker")["returns"].transform(lambda x: x.rolling(window=10, min_periods=1).mean())
#Display the first 5 rows of the dataframe
print(pdf.head())  


             Date   Open   High    Low  Close  Adj Close      Volume   source  \
ticker                                                                          
ACN    2001-07-19  15.10  15.29  15.00  15.17  11.404394  34994300.0  ACN.csv   
ACN    2001-07-20  15.05  15.05  14.80  15.01  11.284108   9238500.0  ACN.csv   
ACN    2001-07-23  15.00  15.01  14.55  15.00  11.276587   7501000.0  ACN.csv   
ACN    2001-07-24  14.95  14.97  14.70  14.86  11.171341   3537300.0  ACN.csv   
ACN    2001-07-25  14.70  14.95  14.65  14.95  11.238999   4208100.0  ACN.csv   

        Year  close_lag1  close_lag2  close_lag3  adj_close_lag1  \
ticker                                                             
ACN     2001         NaN         NaN         NaN             NaN   
ACN     2001       15.17         NaN         NaN       11.404394   
ACN     2001       15.01       15.17         NaN       11.284108   
ACN     2001       15.00       15.01       15.17       11.276587   
ACN     2001       14.86

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It was not necessary to convert to pandas. It would have been better to do it in Dask if the dataset was too big for my computer's memory. 

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.